In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [35]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
labels = {"A":0, "B":1, "C":2, "D":3, "E":4}
train["encoded_label"] = train["answer"].map(labels)
value= train.loc[150, "encoded_label"]
value

np.int64(2)

In [36]:
row0 = train.iloc[0]
formatted_input = str(str(row0["prompt"]) + " [SEP] " + str(row0["B"]))
length = len(formatted_input)
length

407

In [37]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch

prompt = str(row0["prompt"])
options = [str(row0["A"]), str(row0["B"]), str(row0["C"]), str(row0["D"]), str(row0["E"])]
formatted_inputs = [prompt + " [SEP] " + opt for opt in options]
at = AutoTokenizer.from_pretrained("bert-base-uncased")
encodings = at(formatted_inputs,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
input_ids = encodings["input_ids"].unsqueeze(0)
print("Final shape:", input_ids.shape)

Final shape: torch.Size([1, 5, 128])


In [38]:
shape = (16, 5, 128)
total_pos = torch.prod(torch.tensor(shape)).item()
total_positions

10240

In [39]:
logits_shape = (1, 5)
num_logits = logits_shape[1]
num_logits

5

In [40]:
attention_mask = encodings["attention_mask"].unsqueeze(0)
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
labels = torch.tensor([1])
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
loss = outputs.loss
loss.dim()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


0

In [43]:
!pip install torchao==0.16.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.3 MB/s eta 0:00:0000:0100:01


In [45]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(r=8,lora_alpha=16,target_modules=["query", "value"],lora_dropout=0.1,bias="none",task_type=TaskType.SEQ_CLS)
model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_params

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


295681

In [47]:
train =train.head(100)
def preprocess(row):
    prompt = str(row["prompt"])
    options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]
    formatted_inputs = [prompt + " [SEP] " + opt for opt in options]
    
    encodings = tokenizer(formatted_inputs,padding="max_length",truncation=True,max_length=128)
    return {"input_ids": encodings["input_ids"],"attention_mask": encodings["attention_mask"],"labels": ord(row["answer"]) - ord("A")}
dataset = Dataset.from_pandas(train).map(preprocess)
choices=len(dataset[0]["input_ids"])
choices

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

5

In [59]:
train=train.head(32)
dataset = Dataset.from_pandas(train).map(preprocess)
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
model = get_peft_model(model, lora_config)
training_args = TrainingArguments(output_dir="./lora_model",per_device_train_batch_size = 4,gradient_accumulation_steps = 1,max_steps = 4)
trainer = Trainer(model=model,args=training_args,train_dataset=dataset)
trainer.train()

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss


TrainOutput(global_step=4, training_loss=1.6017361879348755, metrics={'train_runtime': 14.6681, 'train_samples_per_second': 1.091, 'train_steps_per_second': 0.273, 'total_flos': 2640170250240.0, 'train_loss': 1.6017361879348755, 'epoch': 0.5})

In [61]:
from peft import PeftModel
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
model = PeftModel.from_pretrained(model,"./lora_model/checkpoint-4" ) 
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
prob_E = probs[0, 4].item()
prob_E

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


0.20202064514160156